# Project 41: Retail Product Demand Estimation and Stockout Risk Analysis-III
## Exploratory Data Analysis, Leakage-Safe Feature Engineering & Model Benchmarking

This notebook provides:
1. Ingestion of multi-product retail sales data (`data/retail_sales.csv`, 5,000+ rows across 16 products).
2. Exploratory Data Analysis (EDA) of category demand, pricing dynamics, promotions, and stockouts.
3. Chronological, leakage-safe feature engineering (lags, rolling statistics shifted by $t-1$, calendar encodings).
4. 3-Model benchmark comparison (Ridge Regression, Random Forest, Gradient Boosting) using `TimeSeriesSplit` and holdout testing.
5. Deterministic stockout risk engine evaluation (Lead Time Demand, Safety Stock, Reorder Point, and exact 3-tier classification: CRITICAL, WARNING / REORDER SOON, HEALTHY).

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
sys.path.append(os.path.abspath('..'))

from data.generate_data import generate_retail_sales_dataset
from pipeline.features import engineer_features, FEATURE_COLUMNS, TARGET_COLUMN
from pipeline.train import run_train_pipeline
from backend.risk_engine import StockoutRiskEngine

### 1. Dataset Generation and Inspection

In [ ]:
data_path = '../data/retail_sales.csv'
if not os.path.exists(data_path):
    generate_retail_sales_dataset(output_path=data_path)

df = pd.read_csv(data_path)
print(f"Total rows: {len(df)}")
print(f"Distinct products: {df['product_id'].nunique()}")
df.head()

### 2. Category Aggregations & Distribution

In [ ]:
cat_summary = df.groupby('category').agg(
    total_sales=('units_sold', 'sum'),
    mean_daily_sales=('units_sold', 'mean'),
    avg_price=('price', 'mean'),
    stockout_days=('stockout_flag', 'sum')
).reset_index()
cat_summary

### 3. Chronological Feature Engineering
Computing leakage-safe shifted features: `lag_1`, `lag_7`, `lag_14`, `rolling_mean_7`, `rolling_std_7`, `rolling_mean_30`.

In [ ]:
features_df = engineer_features(df, fill_lags=True)
features_df[['product_id', 'date', 'units_sold', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_std_7']].tail(10)

### 4. 3-Model Benchmark Comparison & Serialization

In [ ]:
results = run_train_pipeline(data_path=data_path, artifacts_dir='../artifacts')
print("Best model selected:", results['best_model'])
pd.DataFrame(results['models']).T

### 5. Deterministic Risk Engine Verification

In [ ]:
engine = StockoutRiskEngine()
# Scenario 1: CRITICAL
sc1 = engine.evaluate_risk(predicted_daily_demand=25.0, current_stock=50, lead_time_days=7, rolling_std_7=4.0)
print("Scenario 1:", sc1['risk_tier'], "| ROP:", sc1['reorder_point'], "| Reorder Qty:", sc1['reorder_qty'])

# Scenario 2: WARNING / REORDER SOON
sc2 = engine.evaluate_risk(predicted_daily_demand=25.0, current_stock=180, lead_time_days=7, rolling_std_7=4.0)
print("Scenario 2:", sc2['risk_tier'], "| ROP:", sc2['reorder_point'], "| Reorder Qty:", sc2['reorder_qty'])

# Scenario 3: HEALTHY
sc3 = engine.evaluate_risk(predicted_daily_demand=25.0, current_stock=300, lead_time_days=7, rolling_std_7=4.0)
print("Scenario 3:", sc3['risk_tier'], "| ROP:", sc3['reorder_point'], "| Reorder Qty:", sc3['reorder_qty'])